In [2]:
# ========================================================
# ADVANCED AGRICULTURAL ML TRAINING & EVALUATION PIPELINE
# Junior Machine Learning Engineer Internship Project
# ========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("--- Step 1: Data Simulation & Feature Engineering ---")
# Simulated agricultural dataset covering soil, climate, and yield
data = {
    'Nitrogen': [45, 60, 30, 85, 50, 40, 90, 75, 55, 65, 48, 82],
    'Phosphorus': [22, 35, 18, 48, 30, 25, 55, 42, 32, 38, 28, 50],
    'Potassium': [35, 52, 28, 62, 42, 38, 72, 58, 45, 48, 39, 65],
    'Temperature': [26.0, 31.5, 23.0, 34.5, 27.5, 24.0, 32.0, 29.0, 28.5, 30.0, 27.0, 33.5],
    'Humidity': [82, 60, 88, 48, 76, 85, 58, 72, 70, 65, 78, 52],
    'pH_Value': [6.5, 7.1, 5.9, 7.3, 6.7, 6.1, 7.4, 6.6, 6.8, 7.0, 6.6, 7.1],
    'Rainfall': [210, 140, 260, 95, 175, 230, 125, 155, 190, 160, 185, 115],
}

df = pd.DataFrame(data)

# Engineered Feature: Temperature-Humidity Interaction Index
df['Temp_Humidity_Index'] = df['Temperature'] * (df['Humidity'] / 100)
# Engineered Feature: NPK Total Ratio
df['Total_NPK'] = df['Nitrogen'] + df['Phosphorus'] + df['Potassium']

# Target variable (Crop Yield in tons/hectare)
df['Crop_Yield'] = (0.015 * df['Total_NPK']) + (0.002 * df['Rainfall']) - (0.1 * abs(df['pH_Value'] - 6.8)) + np.random.normal(0, 0.1, len(df))

# Features and Target split
X = df.drop(columns=['Crop_Yield'])
y = df['Crop_Yield']

print("\n--- Step 2: Data Partitioning (70% Training, 15% Validation, 15% Testing) ---")
# First split: 70% train, 30% temp (for validation/test split)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
# Second split: dividing 30% equally into validation and testing (15% each)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Training set shape:   {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Testing set shape:    {X_test.shape}")

print("\n--- Step 3: Hyperparameter Tuning (Grid Search Strategy) ---")
# Defining parameter grid for hyperparameter optimization (Week 4 requirement)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Best Hyperparameters Found: {grid_search.best_params_}")

print("\n--- Step 4: Model Evaluation & Error Analysis (Week 5 requirement) ---")
# Predictions on Test Data
predictions = best_model.predict(X_test)

# Calculating Metrics
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"R-squared ($R^2$) Score:        {r2:.4f}")

# Cross-Validation Score
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2')
print(f"5-Fold Cross-Validation Average $R^2$: {cv_scores.mean():.4f}")

print("\n--- Step 5: Residual / Error Analysis ---")
residuals = y_test - predictions
print("Sample Residuals (Actual - Predicted):")
for actual, pred, res in zip(y_test[:3], predictions[:3], residuals[:3]):
    print(f"Actual: {actual:.2f} | Predicted: {pred:.2f} | Error (Residual): {res:.4f}")

print("\nAdvanced Pipeline Executed Successfully!")

--- Step 1: Data Simulation & Feature Engineering ---

--- Step 2: Data Partitioning (70% Training, 15% Validation, 15% Testing) ---
Training set shape:   (8, 9)
Validation set shape: (2, 9)
Testing set shape:    (2, 9)

--- Step 3: Hyperparameter Tuning (Grid Search Strategy) ---
Best Hyperparameters Found: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}

--- Step 4: Model Evaluation & Error Analysis (Week 5 requirement) ---
Root Mean Squared Error (RMSE): 0.0930
Mean Absolute Error (MAE):      0.0829
R-squared ($R^2$) Score:        -0.0504
5-Fold Cross-Validation Average $R^2$: 0.3152

--- Step 5: Residual / Error Analysis ---
Sample Residuals (Actual - Predicted):
Actual: 2.51 | Predicted: 2.63 | Error (Residual): -0.1252
Actual: 2.33 | Predicted: 2.37 | Error (Residual): -0.0406

Advanced Pipeline Executed Successfully!
